In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import DataLoader, random_split, Subset, Dataset
import os
import warnings
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# =============================================================================
# 1. CLASS DEFINITIONS (Unchanged)
# =============================================================================

class ViTTwoHead(nn.Module):
    def __init__(self, num_fruits=5, num_qualities=3):
        super().__init__()
        self.base_model = models.vit_b_16(weights=None)
        num_ftrs = self.base_model.heads.head.in_features
        self.base_model.heads.head = nn.Identity()
        self.fruit_head = nn.Linear(num_ftrs, num_fruits)
        self.qual_head = nn.Linear(num_ftrs, num_qualities)

    def forward(self, x):
        features = self.base_model(x)
        fruit_out = self.fruit_head(features)
        qual_out = self.qual_head(features)
        return fruit_out, qual_out

class FruitVisionDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.fruit_classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.quality_classes = ['fresh', 'rotten', 'formalin-mixed']
        
        for fruit_index, fruit in enumerate(self.fruit_classes):
            fruit_path = os.path.join(root_dir, fruit)
            if not os.path.isdir(fruit_path): continue
            for qual_index, qual in enumerate(self.quality_classes):
                qual_path = os.path.join(fruit_path, qual)
                if not os.path.isdir(qual_path): continue
                for img_file in os.listdir(qual_path):
                    self.samples.append((os.path.join(qual_path, img_file), fruit_index, qual_index))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, fruit_label, qual_label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        plot_image = np.array(image.resize((224, 224))) / 255.0
        if self.transform:
            tensor_image = self.transform(image)
        return tensor_image, plot_image, fruit_label, qual_label

# =============================================================================
# 2. SETUP MODEL AND DATA
# =============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = 'vision_sgd_pretrainedFalse_cosine_model_combined.pth'
DATA_DIR = r'D:\Research\Datasets\FruitVision\Augmented-Resized Image'
IMAGE_SIZE = 224

print(f"Loading model from: {MODEL_PATH}")
model = ViTTwoHead(num_fruits=5, num_qualities=3)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.to(device).eval()
print("Model loaded successfully.")

val_test_transforms = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
test_dataset = FruitVisionDataset(DATA_DIR, transform=val_test_transforms)

# --- MODIFIED THIS LINE ---
# Removed num_workers to disable parallel data loading.
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
print("Test data loaded successfully. Parallel loading is disabled.")

# =============================================================================
# 3. CUSTOM OCCLUSION ANALYSIS FUNCTION
# =============================================================================

def perform_occlusion_analysis(model, input_tensor, true_class_idx, patch_size=16, stride=8, occlusion_value=0):
    """
    Performs a sliding-patch occlusion analysis.
    """
    width, height = input_tensor.shape[-2], input_tensor.shape[-1]
    heatmap = torch.zeros((width, height), device='cpu')

    with torch.no_grad():
        _, original_output = model(input_tensor)
        original_probs = torch.softmax(original_output, dim=-1)
        original_score = original_probs[0, true_class_idx]

    for h in range(0, height, stride):
        for w in range(0, width, stride):
            occluded_tensor = input_tensor.clone()
            h_start, w_start = h, w
            h_end, w_end = min(height, h + patch_size), min(width, w + patch_size)
            occluded_tensor[:, :, h_start:h_end, w_start:w_end] = occlusion_value

            with torch.no_grad():
                _, new_output = model(occluded_tensor)
                new_probs = torch.softmax(new_output, dim=-1)
                new_score = new_probs[0, true_class_idx]
            
            confidence_drop = original_score - new_score
            heatmap[h_start:h_end, w_start:w_end] += confidence_drop.cpu()

    return heatmap.numpy()


def plot_comprehensive_occlusion(model, data_loader, images_per_combo=2):
    fruit_classes = data_loader.dataset.fruit_classes
    quality_classes = data_loader.dataset.quality_classes
    
    images_to_plot = {f_idx: {q_idx: [] for q_idx in range(len(quality_classes))} for f_idx in range(len(fruit_classes))}
    total_combos = len(fruit_classes) * len(quality_classes)
    combos_completed = 0
    desc = f"Scanning for {images_per_combo} images from each of {total_combos} combinations"
    
    for tensor_image, plot_image, fruit_label_t, qual_label_t in tqdm(data_loader, desc=desc):
        fruit_label = fruit_label_t.item()
        qual_label = qual_label_t.item()
        
        if len(images_to_plot[fruit_label][qual_label]) < images_per_combo:
            images_to_plot[fruit_label][qual_label].append({"tensor": tensor_image, "plot_image": plot_image})
            if len(images_to_plot[fruit_label][qual_label]) == images_per_combo:
                combos_completed += 1

        if combos_completed == total_combos:
            tqdm.write("\n-> All required images collected.")
            break
            
    # --- PLOTTING LOGIC ---
    num_rows = total_combos * images_per_combo
    num_cols = 2 # Original, Occlusion
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 5, num_rows * 5))
    fig.suptitle("Comprehensive Occlusion Analysis", fontsize=20)
    
    print("\nGenerating Occlusion Analysis visualizations (this can be slow)...")
    plot_row = 0
    for fruit_idx in sorted(images_to_plot.keys()):
        for qual_idx in sorted(images_to_plot[fruit_idx].keys()):
            items = images_to_plot[fruit_idx][qual_idx]
            for item in items:
                ax_orig, ax_occlusion = axes[plot_row]
                
                input_tensor = item["tensor"].to(device)
                numpy_plot_image = item["plot_image"].squeeze().numpy()
                
                true_class_idx = qual_idx
                
                # --- Generate Occlusion Analysis ---
                occlusion_map = perform_occlusion_analysis(model, input_tensor, true_class_idx)
                
                # Get model's prediction for the title
                with torch.no_grad():
                    _, qual_logits = model(input_tensor)
                predicted_class = qual_logits.argmax().item()

                # Plotting
                fruit_name = fruit_classes[fruit_idx]
                true_name = quality_classes[true_class_idx]
                pred_name = quality_classes[predicted_class]

                ax_orig.imshow(numpy_plot_image)
                ax_orig.set_title(f"{fruit_name} (Original)\nTrue: {true_name}")
                ax_orig.axis('off')
                
                ax_occlusion.imshow(occlusion_map, cmap='jet')
                ax_occlusion.set_title(f"Occlusion (for '{true_name}')\nPredicted: {pred_name}")
                ax_occlusion.axis('off')

                plot_row += 1
            
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# =============================================================================
# 4. RUN THE VISUALIZATION
# =============================================================================
plot_comprehensive_occlusion(model, test_loader, images_per_combo=2)

Loading model from: vision_sgd_pretrainedFalse_cosine_model_combined.pth
Model loaded successfully.
Test data loaded successfully. Parallel loading is disabled.


Scanning for 2 images from each of 15 combinations:  93%|██████████████████████▎ | 68214/73389 [28:32<02:09, 39.83it/s]



-> All required images collected.

Generating Occlusion Analysis visualizations (this can be slow)...


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset
import os
import warnings
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# =============================================================================
# 1. CLASS DEFINITIONS (Unchanged)
# =============================================================================

class ViTTwoHead(nn.Module):
    def __init__(self, num_fruits=5, num_qualities=3):
        super().__init__()
        self.base_model = models.vit_b_16(weights=None)
        num_ftrs = self.base_model.heads.head.in_features
        self.base_model.heads.head = nn.Identity()
        self.fruit_head = nn.Linear(num_ftrs, num_fruits)
        self.qual_head = nn.Linear(num_ftrs, num_qualities)

    def forward(self, x):
        features = self.base_model(x)
        fruit_out = self.fruit_head(features)
        qual_out = self.qual_head(features)
        return fruit_out, qual_out

class FruitVisionDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.fruit_classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.quality_classes = ['fresh', 'rotten', 'formalin-mixed']
        
        for fruit_index, fruit in enumerate(self.fruit_classes):
            fruit_path = os.path.join(root_dir, fruit)
            if not os.path.isdir(fruit_path): continue
            for qual_index, qual in enumerate(self.quality_classes):
                qual_path = os.path.join(fruit_path, qual)
                if not os.path.isdir(qual_path): continue
                for img_file in os.listdir(qual_path):
                    self.samples.append((os.path.join(qual_path, img_file), fruit_index, qual_index))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, fruit_label, qual_label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        plot_image = np.array(image.resize((224, 224))) / 255.0
        if self.transform:
            tensor_image = self.transform(image)
        return tensor_image, plot_image, fruit_label, qual_label

# =============================================================================
# 2. SETUP MODEL AND DATA
# =============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = 'vision_sgd_pretrainedFalse_cosine_model_combined.pth'
DATA_DIR = r'D:\Research\Datasets\FruitVision\Augmented-Resized Image'

print(f"Loading model from: {MODEL_PATH}")
model = ViTTwoHead(num_fruits=5, num_qualities=3)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.to(device).eval()
print("Model loaded successfully.")

val_test_transforms = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
test_dataset = FruitVisionDataset(DATA_DIR, transform=val_test_transforms)

# Simple loader, no workers, no shuffling
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
print("Test data loaded successfully.")

# =============================================================================
# 3. REWRITTEN OCCLUSION ANALYSIS FUNCTION (Memory-Efficient)
# =============================================================================

def perform_occlusion_analysis(model, input_tensor, true_class_idx, patch_size=16, stride=8, occlusion_value=0):
    width, height = input_tensor.shape[-2], input_tensor.shape[-1]
    heatmap = torch.zeros((height, width), device='cpu')
    
    # --- MEMORY FIX: We only clone the tensor ONCE outside the loop ---
    occluded_tensor = input_tensor.clone()

    with torch.no_grad():
        _, original_output = model(input_tensor)
        original_probs = torch.softmax(original_output, dim=-1)
        original_score = original_probs[0, true_class_idx]

        # Iterate over the image with a sliding patch
        for h in tqdm(range(0, height, stride), desc="Occlusion Progress"):
            for w in range(0, width, stride):
                h_start, w_start = h, w
                h_end, w_end = min(height, h + patch_size), min(width, w + patch_size)

                # --- MEMORY FIX: Store original values, occlude, then restore ---
                # This avoids creating a new tensor in every single loop iteration
                original_patch = occluded_tensor[:, :, h_start:h_end, w_start:w_end].clone()
                occluded_tensor[:, :, h_start:h_end, w_start:w_end] = occlusion_value

                _, new_output = model(occluded_tensor)
                new_probs = torch.softmax(new_output, dim=-1)
                new_score = new_probs[0, true_class_idx]
                
                # Restore the patch for the next iteration
                occluded_tensor[:, :, h_start:h_end, w_start:w_end] = original_patch

                confidence_drop = original_score - new_score
                heatmap[h_start:h_end, w_start:w_end] += confidence_drop.cpu()

    return heatmap.numpy()

# =============================================================================
# 4. SIMPLIFIED VISUALIZATION LOOP
# =============================================================================

num_images_to_test = 3
print(f"\n--- Starting Occlusion Analysis for the first {num_images_to_test} images ---")

for i, (tensor_image, plot_image, fruit_label, qual_label) in enumerate(test_loader):
    if i >= num_images_to_test:
        break

    print(f"\nProcessing image {i+1}/{num_images_to_test}...")
    
    input_tensor = tensor_image.to(device)
    numpy_plot_image = plot_image.squeeze() # Remove batch dimension
    true_class_idx = qual_label.item()

    # --- Generate Occlusion Analysis ---
    occlusion_map = perform_occlusion_analysis(model, input_tensor, true_class_idx)
    
    # Get model's prediction for the title
    with torch.no_grad():
        _, qual_logits = model(input_tensor)
    predicted_class = qual_logits.argmax().item()

    # --- Plotting ---
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    fruit_name = test_dataset.fruit_classes[fruit_label.item()]
    true_name = test_dataset.quality_classes[true_class_idx]
    pred_name = test_dataset.quality_classes[predicted_class]

    axes[0].imshow(numpy_plot_image)
    axes[0].set_title(f"{fruit_name} (Original)\nTrue: {true_name}")
    axes[0].axis('off')
    
    im = axes[1].imshow(occlusion_map, cmap='jet')
    axes[1].set_title(f"Occlusion (for '{true_name}')\nPredicted: {pred_name}")
    axes[1].axis('off')
    fig.colorbar(im, ax=axes[1])
    
    plt.show()

Loading model from: vision_sgd_pretrainedFalse_cosine_model_combined.pth
Model loaded successfully.
Test data loaded successfully.

--- Starting Occlusion Analysis for the first 3 images ---

Processing image 1/3...


Occlusion Progress: 100%|██████████████████████████████████████████████████████████████| 28/28 [00:15<00:00,  1.82it/s]
